# Jio Attrition Intelligence System
### Run cells in order — each cell depends on the previous one

In [ ]:
# ============================================================
# CELL 1 — Mount Drive + Install all packages
# Run once per session. Takes 3-5 minutes first time.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/attrition-agent'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

# Install all packages
# groq = fast free LLM API for Colab
# pyngrok = exposes FastAPI to public URL
!pip install -q anthropic langchain langchain-anthropic langgraph \
    chromadb sentence-transformers xgboost scikit-learn shap \
    fastapi uvicorn pydantic faker pandas numpy PyYAML \
    python-dotenv slack-sdk matplotlib seaborn lifelines \
    groq langchain-groq pyngrok

print('\n✅ All packages installed')

In [ ]:
# ============================================================
# CELL 2 — Set API keys
# Get Groq key free at https://console.groq.com
# Get ngrok token free at https://ngrok.com
# ============================================================

import os

# Paste your keys here
os.environ['GROQ_API_KEY'] = 'your_groq_key_here'
os.environ['NGROK_TOKEN']  = 'your_ngrok_token_here'

# Verify Groq connection
from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])
test = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Say OK'}],
    max_tokens=5
)
print(f'Groq connected: {test.choices[0].message.content}')

In [ ]:
# ============================================================
# CELL 3 — Create folder structure
# Run once. Creates all needed directories.
# ============================================================

import os
dirs = [
    'data/synthetic',
    'models',
    'rag/policies',
    'rag/chroma',
    'agents',
    'api',
    'logs'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('✅ Folder structure created')
!ls -la

In [ ]:
# ============================================================
# CELL 4 — Verify all uploaded files exist
# Upload all .py files and config.yaml before running this
# ============================================================

required_files = [
    'config.yaml',
    'data/synthetic/generate.py',
    'data/synthetic/feature_engineering.py',
    'models/train.py',
    'models/ev_scoring.py',
    'rag/build_rag.py',
    'rag/retriever.py',
    'agents/attrition_agent.py',
    'api/main.py',
    'rag/policies/leave_policy.txt',
    'rag/policies/promotion_criteria.txt',
    'rag/policies/retention_guidelines.txt',
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌ MISSING'
    print(f'{status} {f}')
    if not exists:
        all_ok = False

print(f'\n{"✅ All files present" if all_ok else "❌ Upload missing files before proceeding"}')

In [ ]:
# ============================================================
# CELL 5 — Generate synthetic dataset
# Creates 1800 employee records with behavioral features
# ============================================================

%run data/synthetic/generate.py

In [ ]:
# ============================================================
# CELL 6 — Feature engineering
# Engineers 50 features across 6 families + interactions
# ============================================================

%run data/synthetic/feature_engineering.py

In [ ]:
# ============================================================
# CELL 7 — Train models
# XGBoost + LR champion-challenger + Cox PH survival
# Takes 2-3 minutes
# ============================================================

%run models/train.py

In [ ]:
# ============================================================
# CELL 8 — EV scoring (full dataset)
# Scores all 1800 employees with P(attrition) + EV + SHAP
# Takes 3-4 minutes (SHAP computation)
# ============================================================

%run models/ev_scoring.py

In [ ]:
# ============================================================
# CELL 9 — Build RAG
# Embeds HR policy docs into ChromaDB
# Takes 1-2 minutes (sentence-transformers download)
# ============================================================

%run rag/build_rag.py

In [ ]:
# ============================================================
# CELL 10 — Test agent on 3 CRITICAL employees
# Runs full 4-node pipeline and shows LLM recommendations
# ============================================================

%run agents/attrition_agent.py

In [ ]:
# ============================================================
# CELL 11 — Start FastAPI server + ngrok public URL
# Open the printed URL in browser to access Swagger UI
# ============================================================

import threading
import uvicorn
from pyngrok import ngrok
import os

ngrok.set_auth_token(os.environ['NGROK_TOKEN'])

def run_server():
    uvicorn.run('api.main:app', host='0.0.0.0', port=8000, log_level='error')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(3)  # wait for server to start

public_url = ngrok.connect(8000)
print(f'\n✅ API running')
print(f'Swagger UI:  {public_url}/docs')
print(f'Health:      {public_url}/health')
print(f'Weekly:      {public_url}/report/weekly')
print(f'Employee:    {public_url}/explain/JIO10000')

In [ ]:
# ============================================================
# CELL 12 — Quick API test (no browser needed)
# Tests all endpoints directly from notebook
# ============================================================

import requests
import json

BASE = 'http://localhost:8000'

# Test 1: Health
r = requests.get(f'{BASE}/health')
print('Health:', r.json())

# Test 2: Weekly digest
r = requests.get(f'{BASE}/report/weekly')
digest = r.json()
print(f'\nWeekly digest:')
print(f'  Critical: {digest["summary"]["critical_count"]}')
print(f'  High:     {digest["summary"]["high_count"]}')
print(f'  ROI:      {digest["budget_summary"]["net_roi"]}x')

# Test 3: Single employee
import pandas as pd
scored = pd.read_csv('data/synthetic/scored_employees.csv')
test_emp = scored[scored['risk_tier'] == 'CRITICAL'].iloc[0]['employee_id']

r = requests.get(f'{BASE}/explain/{test_emp}')
emp_result = r.json()
print(f'\nEmployee {test_emp}:')
print(f'  Risk tier:  {emp_result["risk_tier"]}')
print(f'  P(leave):   {emp_result["p_attrition"]}')
print(f'  EV:         Rs {emp_result["ev"]:,.0f}')
print(f'  Narrative:  {emp_result["recommendation"].get("narrative", "N/A")}')

# Test 4: Budget simulation
r = requests.get(f'{BASE}/report/budget-sim?correction_type=CRITICAL')
budget = r.json()
print(f'\nBudget simulation (CRITICAL only):')
print(f'  Employees: {budget["employees_affected"]}')
print(f'  Cost:      Rs {budget["total_intervention_cost"]:,.0f}')
print(f'  Avoided:   Rs {budget["replacement_cost_avoided"]:,.0f}')
print(f'  ROI:       {budget["net_roi"]}x')